In [1]:
import pandas as pd

# Create a sample DataFrame with two binary flags and a multiplier
sample_df = pd.DataFrame({
    'model': [3.0, 3.1, 3.2, 4.1, 3.0, 3.1, 3.2, 4.1],
    'flag_a': [1, 0, 1, 0, 0, 1, 0, 1],
    'flag_b': [0, 1, 1, 0, 1, 0, 1, 0],
    'multiplier': [1.0]*8
})

print('Original DataFrame:')
print(sample_df)

# Variation 1: Apply adjustment only where flag_a == 1
sample_df['multiplier_var1'] = sample_df['multiplier']
sample_df.loc[sample_df['flag_a'] == 1, 'multiplier_var1'] *= 1.1

print('\nVariation 1: Only flag_a == 1 adjusted')
print(sample_df[['model', 'flag_a', 'flag_b', 'multiplier_var1']])

# Variation 2: Apply adjustment only where flag_b == 1
sample_df['multiplier_var2'] = sample_df['multiplier']
sample_df.loc[sample_df['flag_b'] == 1, 'multiplier_var2'] *= 1.2
print('\nVariation 2: Only flag_b == 1 adjusted')
print(sample_df[['model', 'flag_a', 'flag_b', 'multiplier_var2']])

# Variation 3: Apply adjustment where both flag_a and flag_b == 1
sample_df['multiplier_var3'] = sample_df['multiplier']
sample_df.loc[(sample_df['flag_a'] == 1) & (sample_df['flag_b'] == 1), 'multiplier_var3'] *= 1.3
print('\nVariation 3: Only both flag_a and flag_b == 1 adjusted')
print(sample_df[['model', 'flag_a', 'flag_b', 'multiplier_var3']])

# Variation 4: Apply adjustment for flag_a == 1, else apply a counter adjustment
sample_df['multiplier_var4'] = sample_df['multiplier']
sample_df.loc[sample_df['flag_a'] == 1, 'multiplier_var4'] *= 1.1
sample_df.loc[sample_df['flag_a'] == 0, 'multiplier_var4'] *= 0.95
print('\nVariation 4: flag_a == 1 gets 1.1, else gets 0.95')
print(sample_df[['model', 'flag_a', 'flag_b', 'multiplier_var4']])

# Variation 5: Apply different adjustments for each model
sample_df['multiplier_var5'] = sample_df['multiplier']
sample_df.loc[sample_df['model'] == 3.0, 'multiplier_var5'] *= 1.05
sample_df.loc[sample_df['model'] == 3.1, 'multiplier_var5'] *= 1.10
sample_df.loc[sample_df['model'] == 3.2, 'multiplier_var5'] *= 1.15
sample_df.loc[sample_df['model'] == 4.1, 'multiplier_var5'] *= 1.20
print('\nVariation 5: Different adjustment for each model')
print(sample_df[['model', 'flag_a', 'flag_b', 'multiplier_var5']])


Original DataFrame:
   model  flag_a  flag_b  multiplier
0    3.0       1       0         1.0
1    3.1       0       1         1.0
2    3.2       1       1         1.0
3    4.1       0       0         1.0
4    3.0       0       1         1.0
5    3.1       1       0         1.0
6    3.2       0       1         1.0
7    4.1       1       0         1.0

Variation 1: Only flag_a == 1 adjusted
   model  flag_a  flag_b  multiplier_var1
0    3.0       1       0              1.1
1    3.1       0       1              1.0
2    3.2       1       1              1.1
3    4.1       0       0              1.0
4    3.0       0       1              1.0
5    3.1       1       0              1.1
6    3.2       0       1              1.0
7    4.1       1       0              1.1

Variation 2: Only flag_b == 1 adjusted
   model  flag_a  flag_b  multiplier_var2
0    3.0       1       0              1.0
1    3.1       0       1              1.2
2    3.2       1       1              1.2
3    4.1       0     

In [ ]:
# Calculate the final multiplier per model (mean of multiplier_var5 for each model)
model_multiplier = sample_df.groupby('model')['multiplier_var5'].mean()
print('Final multiplier per model:')
print(model_multiplier)

# Calculate the weighted average multiplier across all models
model_counts = sample_df['model'].value_counts().sort_index()
weighted_avg_multiplier = (model_multiplier * model_counts).sum() / model_counts.sum()
print('\nWeighted average multiplier across all models:')
print(weighted_avg_multiplier)


In [7]:
# Simulate KMX line of business with 4 models and model-specific adjustments
kmx_df = pd.DataFrame({
    'account_id': range(1, 5),
    'model': [3.0] + [3.1] + [3.2] + [4.1],
    'base_multiplier': [1.0]*4
})

# Add example adjustment columns to kmx_df
kmx_df['fraud_adjustment'] = [1.05, 1.10, 1.00, 1.2]
# kmx_df['driver_flag'] = [1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0]

# Start with base multiplier
kmx_df['loss_multiplier'] = kmx_df['base_multiplier']

leave_out = None  # Change this to exclude an adjustment

if leave_out != 'Fraud':
    kmx_df['loss_multiplier'] *= 1 + (kmx_df['fraud_adjustment'] - 1)

# if leave_out != 'Driver flag':
#     kmx_df['loss_multiplier'] *= 1 + 0.15 * kmx_df['driver_flag']

print('KMX DataFrame after conditional adjustments:')
print(kmx_df)

# Calculate the overall KMX multiplier (weighted average)
overall_kmx_multiplier = kmx_df['loss_multiplier'].mean()
print('\nOverall KMX multiplier (weighted average):')
print(overall_kmx_multiplier)

# If you want to see the average per model as well:
model_means = kmx_df.groupby('model')['base_multiplier'].mean()
print('\nAverage multiplier per model:')
print(model_means)


KMX DataFrame after conditional adjustments:
   account_id  model  base_multiplier  fraud_adjustment  loss_multiplier
0           1    3.0              1.0              1.05             1.05
1           2    3.1              1.0              1.10             1.10
2           3    3.2              1.0              1.00             1.00
3           4    4.1              1.0              1.20             1.20

Overall KMX multiplier (weighted average):
1.0875000000000001

Average multiplier per model:
model
3.0    1.0
3.1    1.0
3.2    1.0
4.1    1.0
Name: base_multiplier, dtype: float64
